# CNTLib: Dataset Preparation

This notebook demonstrates the CNTLib dataset-preparation workflow for the
released CNT dataset.

The published dataset already includes the frozen train/validation/test split
manifests used in the experiments. Therefore, there is no need to regenerate the experimental splits before running
training, inference, or evaluation, except for experimentation purposes.

This notebook is provided to document and demonstrate the dataset-preparation
infrastructure used by CNTLib.

Workflow:

1. Configure the dataset location.
2. Validate the canonical `images/` and `masks/` directories.
3. Run the dataset-preparation workflow.
4. Inspect the resulting metadata, split, and COCO annotation artifacts.

## 1. Setup

In [1]:
from pathlib import Path

import cnt_project

from cnt_project.io.paths import DatasetPaths, OutputPaths

from utils import run_module

### Verify CNTLib installation

The path printed below should point to the installed CNTLib package in the
current notebook environment.

In [2]:
print("CNTLib import package:")
print(cnt_project.__file__)

CNTLib import package:
C:\Users\abd93000\PycharmProjects\cnt_project_review\src\cnt_project\__init__.py


## 2. Configuration

`DATASET_ROOT` points to the canonical CNT dataset.

The dataset contains the source AFM images, instance masks, frozen split
manifests, and derived preprocessing artifacts used by downstream CNTLib
workflows.

In [3]:
# ------------------------------------------------------------------
# User configuration
# ------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "examples":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

DATASET_ROOT = (
    PROJECT_ROOT
    / "data"
    / "cnt_segmentation"
)

SPLIT_NAME = "stratified_seed_0"

### Resolve Canonical Paths

In [4]:
dataset_paths = DatasetPaths.from_root(
    DATASET_ROOT
)

DATASET_ROOT = dataset_paths.root

SPLIT_MANIFEST = dataset_paths.split_manifest_csv(
    SPLIT_NAME
)

SPLIT_METADATA = dataset_paths.split_metadata_yaml(
    SPLIT_NAME
)

### Validate the Input Dataset

The canonical dataset must contain matching source `images/` and `masks/`
directories.

The released dataset additionally contains the frozen split manifests used for
the reported experiments. These manifests should be retained when reproducing
the paper results.

In [5]:
IMAGES_DIR = dataset_paths.images_root
MASKS_DIR = dataset_paths.masks_root

print("Dataset root:")
print(DATASET_ROOT)

print("\nImages directory:")
print(IMAGES_DIR)

print("\nMasks directory:")
print(MASKS_DIR)

print("\nFrozen split manifest:")
print(SPLIT_MANIFEST)


if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset root does not exist: {DATASET_ROOT}"
    )

if not IMAGES_DIR.exists():
    raise FileNotFoundError(
        f"Dataset images directory does not exist: {IMAGES_DIR}"
    )

if not MASKS_DIR.exists():
    raise FileNotFoundError(
        f"Dataset masks directory does not exist: {MASKS_DIR}"
    )

if not SPLIT_MANIFEST.exists():
    raise FileNotFoundError(
        f"Frozen split manifest does not exist: {SPLIT_MANIFEST}"
    )


image_files = [
    path
    for path in IMAGES_DIR.iterdir()
    if path.is_file()
]

mask_files = [
    path
    for path in MASKS_DIR.iterdir()
    if path.is_file()
]

print(f"\nImages found: {len(image_files)}")
print(f"Masks found:  {len(mask_files)}")

if not image_files:
    raise ValueError(
        "The dataset contains no image files."
    )

if not mask_files:
    raise ValueError(
        "The dataset contains no mask files."
    )

print("\nInitial dataset configuration is valid.")

Dataset root:
C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation

Images directory:
C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\images

Masks directory:
C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\masks

Frozen split manifest:
C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\splits\stratified_seed_0.csv

Images found: 130
Masks found:  130

Initial dataset configuration is valid.


## 3. Prepare the Dataset

`prepare_dataset_runner` is CNTLib's high-level dataset-preparation entry point.

The released dataset already includes the frozen experimental split manifests.
Therefore, reproducing the reported experiments does not require generating new
train/validation/test assignments.

In [6]:
run_module(
    "cnt_project.preprocessing.runners.prepare_dataset_runner",
    "--help",
)

Running:
c:\Users\abd93000\PycharmProjects\cnt_project_review\.venv_review\Scripts\python.exe -u -m cnt_project.preprocessing.runners.prepare_dataset_runner --help
--------------------------------------------------------------------------------
usage: prepare_dataset_runner.py [-h] [--dataset-root DATASET_ROOT]
                                 [--split-name SPLIT_NAME]
                                 [--policy {validate,regenerate}]
                                 [--density-method {tertile,kmeans}]
                                 [--density-min-mask-area DENSITY_MIN_MASK_AREA]
                                 [--noise-method {otsu,kmeans}]
                                 [--noise-wavelet NOISE_WAVELET]
                                 [--noise-expected-shape NOISE_EXPECTED_SHAPE]
                                 [--seed SEED]
                                 [--train-fraction TRAIN_FRACTION]
                                 [--val-fraction VAL_FRACTION]
                           

### Build the preparation configuration

This example uses `policy="validate"`.

With this policy, existing valid artifacts are retained and missing downstream
artifacts are generated. This is safer for an introductory workflow than
unconditionally regenerating an already prepared dataset.

In [7]:
prepare_args = [
    "--dataset-root",
    str(DATASET_ROOT),

    "--split-name",
    SPLIT_NAME,

    "--policy",
    "validate",

    "--seed",
    "0",
]

prepare_args

['--dataset-root',
 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_review\\data\\cnt_segmentation',
 '--split-name',
 'stratified_seed_0',
 '--policy',
 'validate',
 '--seed',
 '0']

### Run dataset preparation

In [8]:
run_module(
    "cnt_project.preprocessing.runners.prepare_dataset_runner",
    *prepare_args,
)

Running:
c:\Users\abd93000\PycharmProjects\cnt_project_review\.venv_review\Scripts\python.exe -u -m cnt_project.preprocessing.runners.prepare_dataset_runner --dataset-root C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation --split-name stratified_seed_0 --policy validate --seed 0
--------------------------------------------------------------------------------
COCO RLE annotations saved to C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\COCO_mask\stratified_seed_0\train\annotations_rle.json
COCO RLE annotations saved to C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\COCO_mask\stratified_seed_0\val\annotations_rle.json
COCO RLE annotations saved to C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\COCO_mask\stratified_seed_0\test\annotations_rle.json
Prepared dataset artifacts:
  dataset_root: C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation
  split_name: stra

## 4. Inspect Prepared Dataset Artifacts

After preparation, the dataset remains unsplit physically: images and masks
stay in their canonical directories.

Train, validation, and test membership are defined by the generated split
manifest. Split-specific COCO ground-truth annotations are stored below
`COCO_mask/`.

In [9]:
SPLIT_MANIFEST = dataset_paths.split_manifest_csv(
    SPLIT_NAME
)

SPLIT_METADATA = dataset_paths.split_metadata_yaml(
    SPLIT_NAME
)

print("Split manifest:")
print(SPLIT_MANIFEST)

print("\nSplit metadata:")
print(SPLIT_METADATA)

print("\nMetadata directory:")
print(dataset_paths.metadata_root)

print("\nCOCO directory:")
print(dataset_paths.coco_root)


if not SPLIT_MANIFEST.exists():
    raise FileNotFoundError(
        f"Split manifest was not generated: {SPLIT_MANIFEST}"
    )

if not SPLIT_METADATA.exists():
    raise FileNotFoundError(
        f"Split metadata was not generated: {SPLIT_METADATA}"
    )

print("\nDataset preparation artifacts are available.")

Split manifest:
C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\splits\stratified_seed_0.csv

Split metadata:
C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\splits\stratified_seed_0.yaml

Metadata directory:
C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\metadata

COCO directory:
C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\COCO_mask

Dataset preparation artifacts are available.


### inspect generated preprocessing files

In [10]:
prepared_roots = [
    dataset_paths.metadata_root,
    dataset_paths.splits_root,
    dataset_paths.coco_root,
]

for root in prepared_roots:
    print(f"\n{root.name}/")

    if not root.exists():
        print("  <not present>")
        continue

    for path in sorted(root.rglob("*")):
        if path.is_file():
            print(
                " -",
                path.relative_to(DATASET_ROOT),
            )


metadata/
 - metadata\density_classified_filenames.csv
 - metadata\noise_classification.csv

splits/
 - splits\all_images.csv
 - splits\default_split.csv
 - splits\default_split.yaml
 - splits\legacy_dataloader_seed_42.csv
 - splits\legacy_dataloader_seed_42.yaml
 - splits\stratified_seed_0.csv
 - splits\stratified_seed_0.yaml
 - splits\stratified_seed_1.csv
 - splits\stratified_seed_1.yaml
 - splits\stratified_seed_123.csv
 - splits\stratified_seed_123.yaml
 - splits\stratified_seed_2.csv
 - splits\stratified_seed_2.yaml
 - splits\stratified_seed_42.csv
 - splits\stratified_seed_42.yaml

COCO_mask/
 - COCO_mask\stratified_seed_0\test\annotations_chain_approx_none.json
 - COCO_mask\stratified_seed_0\test\annotations_chain_approx_simple.json
 - COCO_mask\stratified_seed_0\test\annotations_rle.json
 - COCO_mask\stratified_seed_0\train\annotations_chain_approx_none.json
 - COCO_mask\stratified_seed_0\train\annotations_chain_approx_simple.json
 - COCO_mask\stratified_seed_0\train\annotati